# Assignment 4 - Delta Lake Essentials

This notebook is a full solution for the Delta Lake assignment covering Delta table creation, ACID operations, merge, schema evolution, time travel, and validation.


## Objective

Build and maintain a customer transactions table using Delta Lake features in Databricks.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
base_path = "/tmp/delta_demo"


## Phase 1 - Create Initial Delta Table


In [ ]:
data = [
    (1, "C001", "Laptop", 50000),
    (2, "C002", "Mobile", 15000),
    (3, "C003", "Tablet", 20000),
    (4, "C004", "Laptop", 55000)
]
columns = ["id", "customer_id", "product", "amount"]

df = spark.createDataFrame(data, columns)
df.show()

df.write.format("delta").mode("overwrite").save(base_path)

delta_df = spark.read.format("delta").load(base_path)
print("Initial Delta table")
delta_df.show()


## Phase 2 - Insert, Update, Delete


In [ ]:
new_rows = [(5, "C005", "Camera", 30000)]
new_df = spark.createDataFrame(new_rows, columns)
new_df.write.format("delta").mode("append").save(base_path)

delta_table = DeltaTable.forPath(spark, base_path)
delta_table.update(condition = col("id") == 2, set = {"amount": lit(18000)})
delta_table.delete(condition = col("id") == 1)

print("After insert, update, delete")
spark.read.format("delta").load(base_path).orderBy("id").show()


## Phase 3 - MERGE INTO for Incremental Load


In [ ]:
updates = [
    (3, "C003", "Tablet", 22000),
    (6, "C006", "Watch", 8000)
]
updates_df = spark.createDataFrame(updates, columns)

print("Before MERGE count:", spark.read.format("delta").load(base_path).count())

delta_table.alias("target").merge(
    updates_df.alias("source"),
    "target.id = source.id"
).whenMatchedUpdate(set = {
    "customer_id": col("source.customer_id"),
    "product": col("source.product"),
    "amount": col("source.amount")
}).whenNotMatchedInsert(values = {
    "id": col("source.id"),
    "customer_id": col("source.customer_id"),
    "product": col("source.product"),
    "amount": col("source.amount")
}).execute()

merged_df = spark.read.format("delta").load(base_path)
print("After MERGE count:", merged_df.count())
merged_df.orderBy("id").show()


## Phase 4 - Schema Evolution


In [ ]:
updates_with_category = [
    (3, "C003", "Tablet", 23000, "Electronics"),
    (7, "C007", "Headphones", 5000, "Accessories")
]
columns_with_category = ["id", "customer_id", "product", "amount", "category"]
updates_schema_df = spark.createDataFrame(updates_with_category, columns_with_category)

updates_schema_df.write.format("delta").mode("append").option("mergeSchema", "true").save(base_path)

evolved_df = spark.read.format("delta").load(base_path)
print("Schema after evolution")
evolved_df.printSchema()
evolved_df.orderBy("id").show()


## Phase 5 - Time Travel and History


In [ ]:
print("History")
spark.sql(f"DESCRIBE HISTORY delta.`{base_path}`").show(truncate=False)

print("Current version data")
spark.read.format("delta").load(base_path).show()

print("Example time travel to version 0")
spark.read.format("delta").option("versionAsOf", 0).load(base_path).show()


## Optional Restore Example


In [ ]:
%sql
-- Run this in Databricks SQL if you want to restore the table to an old version
-- RESTORE TABLE delta.`/tmp/delta_demo` TO VERSION AS OF 0;


## Phase 6 - Validation


In [ ]:
final_df = spark.read.format("delta").load(base_path)

print("Total row count:", final_df.count())
print("Duplicate ids:")
final_df.groupBy("id").count().filter(col("count") > 1).show()

print("Updated record for id = 3")
final_df.filter(col("id") == 3).show()

print("Check for negative amounts")
final_df.filter(col("amount") < 0).show()

print("Final dataset")
final_df.orderBy("id").show()


## Key Learning Points

- Delta Lake supports ACID-style updates on data lake storage.
- `MERGE` helps combine insert and update logic for incremental loads.
- Schema evolution helps absorb new columns safely.
- Time travel and history make rollback and auditing possible.
